In [12]:
import pandas as pd
import os
from discovery_utils import PROJECT_DIR
from discovery_utils.getters import crunchbase
from discovery_utils.utils import viz_landscape

In [10]:
import importlib
importlib.reload(crunchbase)
importlib.reload(viz_landscape)

[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/karlis.kanders/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


<module 'discovery_utils.utils.viz_landscape' from '/Users/karlis.kanders/Code/discovery_utils/discovery_utils/utils/viz_landscape.py'>

In [ ]:
CB = crunchbase.CrunchbaseGetter(vector_db_path=PROJECT_DIR / "tmp/vector_db")

2025-01-22 17:14:52,336 - discovery_utils.getters.crunchbase - INFO - Checking for latest version of data in S3 bucket: discovery-iss
2025-01-22 17:14:52,451 - discovery_utils.getters.crunchbase - INFO - Latest Crunchbase version found: Crunchbase_2025-01-20


In [ ]:
# Get all companies in the provided Crunchbase categories
list_of_categories = ["Diabetes"]
selected_df = CB.get_companies_in_categories(["Diabetes"], category_type="narrow")

matching_ids = set(list(selected_df.id.to_list()))

2025-01-22 17:15:31,479 - discovery_utils.getters.crunchbase - INFO - Downloading parquet file: data/crunchbase/enriched/organizations_full.parquet
2025-01-22 17:15:55,889 - discovery_utils.getters.crunchbase - INFO - Successfully downloaded and read parquet file: data/crunchbase/enriched/organizations_full.parquet


In [6]:
# Find matching companies in the vector database
id_condition = "id in ('{}')".format("', '".join(list(matching_ids)))
vectors_df = CB.VectorDB.vector_db.search().where(id_condition).limit(30000).to_pandas()

In [ ]:
# Generate the landscape visualisation
fig, cb_viz_df = viz_landscape.generate_crunchbase_landscape(vectors_df, CB)

2025-01-22 17:20:02,861 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-01-22 17:20:04,227 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-01-22 17:20:05,350 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-01-22 17:20:06,340 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-01-22 17:20:07,478 - httpx - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
2025-01-22 17:20:07,534 - root - INFO - Outliers were successfully reduced


In [ ]:
# Output the HTML chart
output_path = "test.html"
fig.save(str(output_path))

In [ ]:
# Clean up
os.remove("test.html")